# Project 5 — Europe Under Pressure: GVC Position and Reshoring Vulnerability

**Author:** Poonum Malhi  
**Series:** Europe Under Pressure (Part 5)  
**Date:** June 2026

## Research Question
> Which EU-27 countries are most exposed to reshoring risk, based on their depth of integration in Global Value Chains (2020)?

## Motivation
The COVID-19 pandemic, the Russia-Ukraine war, and rising US-China tensions have triggered a global debate about reshoring — bringing supply chains back home. But reshoring is not painless. Countries deeply embedded in GVCs face the greatest disruption if global production networks shorten.

This project builds a **Reshoring Vulnerability Index** for all 27 EU member states, combining:
1. **GVC depth** — how much of a country's value added flows into global production networks (OECD TiVA 2023)
2. **Manufacturing intensity** — the share of GDP from manufacturing (World Bank 2020)

## Papers this builds on
- **Antràs & Chor (2013)** — *Organizing the Global Value Chain*, Econometrica: framework for upstream/downstream positioning
- **Felbermayer et al. (2024)** — on reshoring costs in European economies
- Professor's own working paper: *The True Cost of Reshoring* (Paris 1)

## Data Sources
- **OECD TiVA 2023** — Forward linkages (domestic value added in world final demand), 2020
- **World Bank** — Manufacturing value added (% of GDP), 2020

## Step 1 — Import libraries

`pandas` handles our data tables. `numpy` handles math operations. `plotly` makes interactive charts.

**Why these and not others?** pandas is the standard for tabular data in economics research. plotly makes charts you can hover over — useful for presentations.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print('Libraries loaded successfully')

## Step 2 — Define our countries and supplementary data

We define the EU-27 country codes (ISO 3-letter standard used by OECD), GDP, and manufacturing shares.

**Why manual GDP data?** The TiVA file gives us value added in USD millions. To compare countries fairly, we need to normalise by economic size — otherwise Germany always looks biggest simply because it's the largest economy.

In [ ]:
# EU-27 country codes and names
eu27 = {
    'AUT':'Austria','BEL':'Belgium','BGR':'Bulgaria','HRV':'Croatia',
    'CYP':'Cyprus','CZE':'Czechia','DNK':'Denmark','EST':'Estonia',
    'FIN':'Finland','FRA':'France','DEU':'Germany','GRC':'Greece',
    'HUN':'Hungary','IRL':'Ireland','ITA':'Italy','LVA':'Latvia',
    'LTU':'Lithuania','LUX':'Luxembourg','MLT':'Malta','NLD':'Netherlands',
    'POL':'Poland','PRT':'Portugal','ROU':'Romania','SVK':'Slovakia',
    'SVN':'Slovenia','ESP':'Spain','SWE':'Sweden'
}

# GDP in USD billions (World Bank 2020)
gdp = {'DEU':3846,'FRA':2630,'ITA':1897,'ESP':1281,'NLD':913,'POL':596,
       'SWE':541,'BEL':524,'AUT':433,'DNK':356,'FIN':271,'IRL':425,
       'ROU':249,'CZE':246,'PRT':228,'GRC':188,'HUN':156,'SVK':105,
       'LUX':73,'BGR':69,'LTU':57,'HRV':57,'SVN':53,'LVA':34,
       'EST':31,'CYP':24,'MLT':14}

# Manufacturing share of GDP % (World Bank 2020)
manuf = {'DEU':19,'FRA':10,'ITA':16,'ESP':13,'NLD':12,'POL':18,
         'SWE':15,'BEL':13,'AUT':17,'DNK':14,'FIN':18,'IRL':22,
         'ROU':21,'CZE':25,'PRT':12,'GRC':9,'HUN':22,'SVK':22,
         'LUX':5,'BGR':16,'LTU':17,'HRV':13,'SVN':21,'LVA':11,
         'EST':17,'CYP':5,'MLT':11}

print(f'EU-27 countries defined: {len(eu27)}')

## Step 3 — Load and clean OECD TiVA data

**What is Forward Linkage?**  
A country's forward linkage measures how much of its domestic value added ends up being used in *other countries' production*. A high forward linkage means your country is supplying inputs to the rest of the world — you are upstream.

**Filter logic:**
- `VALUE_ADDED_SOURCE_AREA` = the country producing the value (our EU-27)
- `FINAL_DEMAND_AREA = 'W'` = world (we want the global total, not bilateral)
- `TIME_PERIOD = 2020` = our reference year

In [ ]:
# Load the OECD TiVA file
df_raw = pd.read_csv('OECD_STI_PIE_DSD_TIVA_FDVA_DF_FDVA_1_1____T_W__T__A.csv')

print('Columns:', df_raw.columns.tolist())
print('Years available:', sorted(df_raw['TIME_PERIOD'].unique()))
print('Total rows:', len(df_raw))

In [ ]:
# Filter: EU-27 countries, world as final demand destination, year 2020
df = df_raw[
    (df_raw['VALUE_ADDED_SOURCE_AREA'].isin(eu27.keys())) &
    (df_raw['FINAL_DEMAND_AREA'] == 'W') &
    (df_raw['TIME_PERIOD'] == 2020)
][['VALUE_ADDED_SOURCE_AREA', 'OBS_VALUE']].copy()

df.columns = ['code', 'forward_linkage_usd_mn']

# Add country names, GDP and manufacturing share
df['country']  = df['code'].map(eu27)
df['gdp_bn']   = df['code'].map(gdp)
df['manuf_pct']= df['code'].map(manuf)

print(f'Countries loaded: {len(df)}')
df.head(5)

## Step 4 — Build the Reshoring Vulnerability Index

**Why two components?**

GVC depth alone tells us how integrated a country is in global chains. But a country with deep GVC integration in *services* (like Luxembourg's finance sector) faces very different reshoring risk than one integrated through *manufacturing* (like Czechia's auto parts sector). Physical goods can be reshored; financial services cannot.

So our index = **GVC depth × manufacturing intensity**

This captures: countries that are both deeply embedded in GVCs *and* do so through manufacturing — the most reshoring-exposed group.

In [ ]:
# GVC depth = forward linkage / GDP (normalised, same USD units)
# forward_linkage is in USD millions, gdp_bn is in USD billions
df['gvc_depth'] = df['forward_linkage_usd_mn'] / (df['gdp_bn'] * 1000)

# Raw vulnerability = GVC depth × manufacturing share
df['vulnerability_raw'] = df['gvc_depth'] * df['manuf_pct']

# Normalise to 0-100 scale for readability
df['reshoring_vulnerability'] = (
    (df['vulnerability_raw'] - df['vulnerability_raw'].min()) /
    (df['vulnerability_raw'].max() - df['vulnerability_raw'].min()) * 100
).round(1)

df = df.sort_values('reshoring_vulnerability', ascending=False).reset_index(drop=True)
df[['country','gvc_depth','manuf_pct','reshoring_vulnerability']]

## Step 5 — Visualise: Reshoring Vulnerability Ranking

In [ ]:
fig = px.bar(
    df.sort_values('reshoring_vulnerability'),
    x='reshoring_vulnerability',
    y='country',
    orientation='h',
    color='reshoring_vulnerability',
    color_continuous_scale='RdYlGn_r',
    title='Reshoring Vulnerability Index — EU-27 (2020)<br><sup>Higher = more exposed to supply chain shortening</sup>',
    labels={'reshoring_vulnerability': 'Vulnerability Index (0-100)', 'country': ''},
    text='reshoring_vulnerability'
)
fig.update_traces(texttemplate='%{text}', textposition='outside')
fig.update_layout(height=750, showlegend=False, coloraxis_showscale=False)
fig.show()

## Step 6 — Scatter: GVC Depth vs Manufacturing Intensity

This chart lets us see *why* each country scores the way it does — and spot outliers.

In [ ]:
fig2 = px.scatter(
    df,
    x='gvc_depth',
    y='manuf_pct',
    size='reshoring_vulnerability',
    color='reshoring_vulnerability',
    color_continuous_scale='RdYlGn_r',
    text='code',
    title='GVC Depth vs Manufacturing Intensity — EU-27 (2020)<br><sup>Bubble size = Reshoring Vulnerability Index</sup>',
    labels={
        'gvc_depth': 'GVC Depth (Forward Linkage / GDP)',
        'manuf_pct': 'Manufacturing Share of GDP (%)'
    }
)
fig2.update_traces(textposition='top center')
fig2.update_layout(height=550)
fig2.show()

## Step 7 — Key Findings

### Most vulnerable countries (Reshoring Index > 70)
- **Czechia (100)** — highest manufacturing share in EU-27 (25% of GDP) combined with deep GVC integration, primarily through German automotive supply chains
- **Ireland (89)** — deeply integrated through pharma and tech GVCs
- **Slovakia (87) and Hungary (81)** — Central European manufacturing hubs embedded in German/Austrian production networks
- **Romania (81) and Slovenia (79)** — similar CEE manufacturing exposure

### Least vulnerable (Reshoring Index < 25)
- **Luxembourg (0.4) and Cyprus (0)** — GVC integration is almost entirely through financial services, not manufacturing. Reshoring of financial flows is far less feasible than reshoring physical goods
- **Greece (18) and France (25)** — relatively low manufacturing shares

### The Germany paradox
Germany scores 71 — high but not top. It has a large manufacturing base but its GVC depth, while large in absolute terms, is moderate relative to GDP. Germany also *anchors* GVCs rather than depending on them, giving it more adjustment capacity.

## Step 8 — Limitations (honest assessment)

1. **Manufacturing share is sector-level, not GVC-specific** — a country with high manufacturing may produce mostly for domestic consumption, not GVCs
2. **Single year (2020) is a COVID year** — GVC participation was disrupted; 2019 would be a cleaner baseline
3. **No backward linkage** — we only measure how much a country feeds into others (forward), not how much it depends on foreign inputs (backward). A complete vulnerability measure needs both
4. **No regression analysis** — this is a descriptive index, not a causal estimate

## Next steps / Part 6
- Add backward linkages to build a fuller GVC dependency measure
- Use 2019 data to avoid COVID distortions  
- Test whether higher vulnerability correlates with larger trade balance swings post-2020
- Replicate Antràs & Chor (2013) upstreamness measure using ICIO tables